# Embedding-Based Dictionary Expansion: Innovation Seeds

This notebook is a Google Colab teaching demo for a finance/accounting NLP workshop.

The exercise is inspired by Li, Mai, Shen, and Yan (2021, RFS), **"Measuring Corporate Culture Using Machine Learning"**. It is **not** a full replication. Instead, it demonstrates the core intuition of embedding-based dictionary expansion:

1. Start with a small set of seed words for a concept, here **innovation**.
2. Train Word2Vec on a domain corpus, here 2024Q4 US earnings-call Q&A turns.
3. Find words close to the average seed vector.
4. Manually inspect candidate words.
5. Compare domain-trained Word2Vec with pre-trained GloVe.

## 1. Setup

If you run this in Google Colab, upload or clone the project folder so that the notebook can see:

- `data/raw/003_all_US_calls_2024Q4.csv`
- `scripts/prepare_w2v_demo_artifacts.py`
- `requirements_colab.txt`

The local Dropbox version also falls back to `../data/003_all_US_calls_2024Q4.csv`.

In [1]:
from pathlib import Path
import os
import random
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

# Find project root. In Colab this is usually /content/<project>; locally this
# notebook lives in Thu_morning/notebooks, so parent is the project root.
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/Thu_morning"),
    Path("/content/Sheffield_NLP_teaching/Thu_morning"),
]
ROOT = next((c for c in candidates if (c / "scripts").exists()), Path.cwd())
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

print("Project root:", ROOT)

req = ROOT / "requirements_colab.txt"
if IN_COLAB:
    if req.exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy<2", "gensim", "scipy<1.13", "nltk", "tqdm"])

sys.path.insert(0, str(ROOT / "scripts"))

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser

import prepare_w2v_demo_artifacts as demo

random.seed(demo.SEED)
np.random.seed(demo.SEED)

demo.ensure_directories(ROOT)
print("Using Word2Vec config:", demo.W2V_CONFIG)

Project root: C:\Users\Helen\Dropbox\Sheffield_NLP_teaching\Thu_morning


Using Word2Vec config: {'vector_size': 50, 'window': 5, 'min_count': 10, 'sg': 1, 'negative': 5, 'epochs': 3, 'seed': 42}


C:\Users\Helen\AppData\Local\Temp\w2v_demo_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Data

We load `data/raw/003_all_US_calls_2024Q4.csv` if available. The code automatically detects the main text column by trying likely names:

`["text", "content", "transcript", "componenttext", "speech", "turn_text", "qa_text", "sentence"]`

If none are found, it prints all columns and asks you to set `TEXT_COL` manually.

In [2]:
RAW_CSV = demo.find_input_csv(ROOT)
print("Input CSV:", RAW_CSV)

TEXT_COL = None  # set manually here if auto-detection fails, e.g. TEXT_COL = "turn_text"
df, text_col = demo.load_qna_data(RAW_CSV, text_col=TEXT_COL)

print("Shape after dropping missing/short text:", df.shape)
print("Detected text column:", text_col)
print("Columns:", list(df.columns))

for col in ["companyid", "companyname", "transcriptid", "speaker_name", "speaker_type"]:
    if col in df.columns:
        print(f"Unique {col}: {df[col].nunique():,}")

df[[c for c in ["companyname", "transcriptid", "speaker_name", "speaker_type", text_col] if c in df.columns]].head()

Input CSV: C:\Users\Helen\Dropbox\Sheffield_NLP_teaching\Thu_morning\data\raw\003_all_US_calls_2024Q4.csv


Shape after dropping missing/short text: (65341, 12)
Detected text column: turn_text
Columns: ['companyname', 'companyid', 'transcriptid', 'call_date', 'headline', 'turn_index', 'transcriptcomponentid', 'transcriptcomponenttypeid', 'component_type', 'speaker_name', 'speaker_type', 'turn_text']
Unique companyid: 1,946
Unique companyname: 1,946
Unique transcriptid: 1,946
Unique speaker_name: 7,744
Unique speaker_type: 4


,companyname,transcriptid,speaker_name,speaker_type,turn_text
0,"10x Genomics, Inc.",3292745,Tejas Savant,Analysts,Perhaps maybe I'll start with one on the quart...
1,"10x Genomics, Inc.",3292745,Adam Taich,Executives,"Let me take a stab at it. So on the Q4 guide, ..."
2,"10x Genomics, Inc.",3292745,Tycho Peterson,Analysts,I'm going to start with instruments. You said ...
3,"10x Genomics, Inc.",3292745,Serge Saxonov,Executives,"Yes. Thanks, Tycho. So maybe on -- first of al..."
4,"10x Genomics, Inc.",3292745,Douglas Schenkel,Analysts,"So as many of you know, I'm a New England Patr..."


## 3. Lightweight Cleaning for Teaching

The full Li et al. pipeline uses Stanford CoreNLP, lemmatisation, NER replacement, dependency parsing for multiword expressions and compounds, stopword removal, Gensim phrase detection, and Word2Vec.

For this classroom demo, we use a lightweight tokenizer that:

- lowercases text
- lightly expands common contractions
- keeps alphabetic tokens and selected hyphenated terms such as `ai-driven`
- removes punctuation and pure numbers
- removes one-letter tokens
- removes common English stopwords
- keeps finance-relevant words such as `may`, `growth`, `margin`, `cash`, `risk`

**Important:** these are Word2Vec-style tokens: words or phrase tokens. They are not LLM subword tokens.

In [3]:
# The function is defined in scripts/prepare_w2v_demo_artifacts.py so the
# notebook and the standalone script use exactly the same cleaning logic.
clean_and_tokenize = demo.clean_and_tokenize

example_text = df[text_col].iloc[0]
print("Original text snippet:")
print(example_text[:500])
print("\nCleaned tokens:")
print(clean_and_tokenize(example_text)[:80])

Original text snippet:
Perhaps maybe I'll start with one on the quarter and the outlook here. So EMEA and APAC held up a lot better for you in the quarter relative to the Americas and you highlighted the bulk of the commercial restructuring was focused on the Americas. So that makes me feel that it's mainly the commercial reorg that was the driver of the $50 million cut, versus incremental macro deterioration relative to your prior guide. So can you just confirm that?\

Cleaned tokens:
['perhaps', 'maybe', 'start', 'one', 'quarter', 'outlook', 'emea', 'apac', 'held', 'lot', 'better', 'quarter', 'relative', 'americas', 'highlighted', 'bulk', 'commercial', 'restructuring', 'focused', 'americas', 'makes', 'feel', 'mainly', 'commercial', 'reorg', 'driver', 'million', 'cut', 'versus', 'incremental', 'macro', 'deterioration', 'relative', 'prior', 'guide', 'confirm']


## 4. Phrase Detection

We train Gensim `Phrases` and `Phraser` objects to create phrase-aware tokens such as:

- `customer_experience`
- `cash_flow`
- `gross_margin`

Defaults: `min_count=10`, `threshold=10`.

The notebook caches processed sentence files:

- `data/processed/w2v_sentences_unigram.txt`
- `data/processed/w2v_sentences_trigram.txt`

If these files already exist, they are loaded instead of rebuilt.

In [4]:
unigram_path = ROOT / "data/processed/w2v_sentences_unigram.txt"
trigram_path = ROOT / "data/processed/w2v_sentences_trigram.txt"

if unigram_path.exists() and trigram_path.exists():
    print("Loading cached tokenized sentences.")
    tokenized = demo.load_sentences(unigram_path)
else:
    tokenized = demo.tokenize_texts(df[text_col])

unigram_sentences, trigram_sentences, bigram, trigram = demo.prepare_phrase_sentences(
    tokenized, ROOT, force=False
)

print("Unigram turns:", len(unigram_sentences))
print("Trigram/phrase-aware turns:", len(trigram_sentences))
print("Example phrase-aware tokens:")
print(trigram_sentences[0][:80])

Loading cached tokenized sentences.


Loading cached processed sentence files and phrase models.


Unigram turns: 65334
Trigram/phrase-aware turns: 65334
Example phrase-aware tokens:
['perhaps', 'maybe', 'start', 'one', 'quarter', 'outlook', 'emea', 'apac', 'held', 'lot', 'better', 'quarter', 'relative', 'americas', 'highlighted', 'bulk', 'commercial', 'restructuring', 'focused', 'americas', 'makes', 'feel', 'mainly', 'commercial', 'reorg', 'driver', 'million', 'cut', 'versus', 'incremental', 'macro', 'deterioration', 'relative', 'prior', 'guide', 'confirm']


## 5. Train Word2Vec

For this workshop demo, we use the settings requested for a quick Colab run:

- `vector_size=50`
- `window=5`
- `min_count=10`
- `sg=1` (skip-gram)
- `negative=5`
- `epochs=3`
- `seed=42`

If the model files already exist, the notebook loads them instead of retraining:

- `models/w2v_2024q4_qna_demo.model`
- `models/w2v_2024q4_qna_demo.kv`

In [5]:
model = demo.train_or_load_w2v(trigram_sentences, ROOT, force=False)

total_tokens = sum(len(sent) for sent in trigram_sentences)
print("Number of sentences/turns:", f"{len(trigram_sentences):,}")
print("Total tokens:", f"{total_tokens:,}")
print("Vocabulary size:", f"{len(model.wv):,}")

Loading cached Word2Vec model.
Number of sentences/turns: 65,334
Total tokens: 2,019,098
Vocabulary size: 13,056


## 6. Seed Words

We begin with a narrow innovation seed list:

In [6]:
innovation_seeds = [
    "innovation", "innovative", "innovate",
    "innovating", "innovations", "creative"
]

covered, missing = demo.covered_seeds(model.wv, innovation_seeds)
print("Covered seeds:", covered)
print("Missing seeds:", missing)
if len(covered) < 3:
    print("WARNING: fewer than 3 seeds are covered; continuing anyway.")

Covered seeds: ['innovation', 'innovative', 'innovate', 'innovating', 'innovations', 'creative']
Missing seeds: []


## 7. Nearest Neighbours in Domain-Trained Word2Vec

We average the vectors for the covered innovation seed words, compute cosine similarity to every vocabulary term, exclude the seeds themselves, and save the top 20 and top 50 neighbours.

In [7]:
w2v_top20, w2v_top50 = demo.save_word2vec_neighbors(model, ROOT)

display(w2v_top20)
print("Saved:")
print(ROOT / "outputs/embedding_demo/innovation_top20_word2vec.csv")
print(ROOT / "outputs/embedding_demo/innovation_top50_word2vec.csv")

Covered seeds (word2vec_2024q4_qna): ['innovation', 'innovative', 'innovate', 'innovating', 'innovations', 'creative']
Missing seeds (word2vec_2024q4_qna): []
Covered seeds (word2vec_2024q4_qna): ['innovation', 'innovative', 'innovate', 'innovating', 'innovations', 'creative']
Missing seeds (word2vec_2024q4_qna): []


,rank,word,similarity,source
0,1,product_suite,0.967082,word2vec_2024q4_qna
1,2,product_offering,0.960936,word2vec_2024q4_qna
2,3,scalable,0.957898,word2vec_2024q4_qna
3,4,blueprint,0.957152,word2vec_2024q4_qna
4,5,differentiating,0.954463,word2vec_2024q4_qna
5,6,tech_stack,0.953393,word2vec_2024q4_qna
6,7,customization,0.951195,word2vec_2024q4_qna
7,8,brand_awareness,0.951025,word2vec_2024q4_qna
8,9,sales_organization,0.950505,word2vec_2024q4_qna
9,10,products_services,0.949292,word2vec_2024q4_qna


Saved:
C:\Users\Helen\Dropbox\Sheffield_NLP_teaching\Thu_morning\outputs\embedding_demo\innovation_top20_word2vec.csv
C:\Users\Helen\Dropbox\Sheffield_NLP_teaching\Thu_morning\outputs\embedding_demo\innovation_top50_word2vec.csv


## 8. Compare with Pre-Trained GloVe

GloVe knows general English from a broad external corpus. We use the same innovation seed list and the same average-seed-vector method.

If the download fails, the section prints a clear message and can be skipped in class.

In [8]:
glove_top20, glove_top50 = demo.save_glove_neighbors(ROOT)

if glove_top20 is not None:
    display(glove_top20)
else:
    print("GloVe download failed. This section can be skipped in class or run when internet is available.")

Loading GloVe glove-wiki-gigaword-100 via gensim.downloader.


Covered seeds (glove_wiki_gigaword_100): ['innovation', 'innovative', 'innovate', 'innovating', 'innovations', 'creative']
Missing seeds (glove_wiki_gigaword_100): []
Covered seeds (glove_wiki_gigaword_100): ['innovation', 'innovative', 'innovate', 'innovating', 'innovations', 'creative']
Missing seeds (glove_wiki_gigaword_100): []


,rank,word,similarity,source
0,1,creativity,0.732125,glove_wiki_gigaword_100
1,2,entrepreneurial,0.711666,glove_wiki_gigaword_100
2,3,technological,0.696194,glove_wiki_gigaword_100
3,4,technologies,0.684928,glove_wiki_gigaword_100
4,5,collaborative,0.673394,glove_wiki_gigaword_100
5,6,entrepreneurship,0.662284,glove_wiki_gigaword_100
6,7,inventive,0.654796,glove_wiki_gigaword_100
7,8,technology,0.647205,glove_wiki_gigaword_100
8,9,utilize,0.641140,glove_wiki_gigaword_100
9,10,designing,0.636595,glove_wiki_gigaword_100


## 9. Compare with a Li-Style Innovation Reference List

This is a small manually entered reference list, not the full Li et al. dictionary. It is used only to illustrate exact and near-overlap checks.

In [9]:
li_innovation_reference = demo.LI_INNOVATION_REFERENCE
print("Reference terms:")
print(li_innovation_reference)

overlap = demo.save_overlap_comparison(ROOT, w2v_top20, glove_top20)
display(overlap)
print("Saved:", ROOT / "outputs/embedding_demo/innovation_overlap_comparison.csv")

Reference terms:
['technology', 'operational_excellence', 'product_innovation', 'customer_experience', 'agility', 'efficient', 'cutting-edge', 'agile', 'customer-centric', 'value_proposition', 'reinvent', 'efficiency', 'customer_value', 'user_experience', 'create', 'pride', 'automation', 'platform', 'innovator', 'go-to-market', 'workflow', 'scalable', 'artificial_intelligence', 'machine_learning', 'resonate', 'newness', 'inventive']


,source,exact_overlap_count,exact_overlap_words,near_overlap_count,near_overlap_matches
0,word2vec_top20,3,artificial_intelligence; resonate; scalable,0,
1,glove_top20,3,cutting_edge; inventive; technology,0,


Saved: C:\Users\Helen\Dropbox\Sheffield_NLP_teaching\Thu_morning\outputs\embedding_demo\innovation_overlap_comparison.csv


## 10. Teaching Interpretation

Key points for students:

- **GloVe knows general English.** It was trained on broad, external text and often returns semantically general neighbours.
- **Word2Vec trained on earnings calls learns earnings-call language.** It may surface terms that are locally meaningful in Q&A discussions, such as product, platform, customer, AI, automation, margin, or workflow terms.
- **A small one-quarter corpus is enough for demonstration, not publication.** The corpus is too small and too narrow to create a final research dictionary.
- **Embedding output is a candidate list, not the final dictionary.** The researcher must inspect, clean, validate, and document choices.
- **This differs from Li et al.** The full method uses heavier preprocessing, including Stanford CoreNLP, lemmatisation, NER replacement, dependency parsing, compound handling, stopword removal, phrase detection, and Word2Vec over a much larger corpus.

## 11. Optional Scoring Exercise

Students can choose 10–20 accepted innovation words after manual inspection of the top 50 neighbours. The simple score below counts accepted words divided by total tokens in each Q&A turn, then aggregates when firm/speaker columns exist.

In [10]:
# Edit this list after inspecting w2v_top50 and/or glove_top50.
accepted_innovation_words = [
    "innovation",
    "innovative",
    "creative",
    "technology",
    "platform",
    "automation",
    "workflow",
    "scalable",
    "efficient",
    "efficiency",
]

scores_by_firm = demo.optional_scoring_exercise(
    df, text_col, ROOT, accepted_words=accepted_innovation_words
)

display(scores_by_firm.head(20))
print("Saved:", ROOT / "outputs/embedding_demo/simple_innovation_scores_by_firm.csv")

,companyname,companyid,speaker_type,transcriptid,turns,innovation_word_hits,token_count,innovation_score
0,"10x Genomics, Inc.",223288117,Analysts,3292745,16,0,596,0.000000
1,"10x Genomics, Inc.",223288117,Executives,3292745,15,3,674,0.004451
2,"1stdibs.Com, Inc.",129622619,Analysts,3304161,4,0,175,0.000000
3,"1stdibs.Com, Inc.",129622619,Executives,3304161,5,1,214,0.004673
4,3D Systems Corporation,308402,Analysts,3319624,13,1,385,0.002597
5,3D Systems Corporation,308402,Executives,3319624,14,6,875,0.006857
6,3M Company,289194,Analysts,3286229,28,3,707,0.004243
7,3M Company,289194,Executives,3286229,32,1,960,0.001042
8,908 Devices Inc.,216496942,Analysts,3307753,8,0,270,0.000000
9,908 Devices Inc.,216496942,Executives,3307753,11,2,772,0.002591


Saved: C:\Users\Helen\Dropbox\Sheffield_NLP_teaching\Thu_morning\outputs\embedding_demo\simple_innovation_scores_by_firm.csv
